# Layout Examples

Ways to arrange the annotator for different tasks. Every example uses the same base config
([`config/layout-and-buttons.yaml`](annotator_config/config/layout-and-buttons.yaml): the demo
detections, `audio_url` audio, an output file in `outputs/`) and passes only its layout settings,
so each cell shows exactly what changes.

1. [Default: form below the spectrogram](#1.-Default:-form-below-the-spectrogram)
2. [Side form](#2.-Side-form) — `side_form: true`, `side_form_split`, a thin button column
3. [Focused review](#3.-Focused-review) — side form, playlist collapsed, no info card
4. [Starting collapsed](#4.-Starting-collapsed) — `*_open`, section titles
5. [Form layouts: row / column / grid](#5.-Form-layouts:-row-/-column-/-grid)
6. [Narrow widgets](#6.-Narrow-widgets) — the side form stacks under the spectrogram

**Tips:** every section's ▾ collapses it; drag the divider between a side form and the
spectrogram; `k` / `j` = next / previous clip; `Shift+Tab` cycles through the sections.

In [2]:
from jupyter_bioacoustic import BioacousticAnnotator

CONFIG = 'annotator_config/config/layout-and-buttons.yaml'


def jba(**settings):
    """Open the annotator on the base config plus ``settings``; print any validation problems."""
    ba = BioacousticAnnotator(config=CONFIG, **settings)
    for msg in ba.validate()['errors']:
        print('❌', msg)
    ba.open()
    return ba


# a small one-click review form, reused below
QUICK_REVIEW = [
    {'row': {'column': 'is_valid', 'items': [
        {'button': {'show_hot_key': True, 'value': 'yes', 'icon': 'jba:check', 'hot_key': 'y', 'required': True}},
        {'button': {'value': 'no', 'icon': 'jba:x', 'hot_key': 'n'}},
        {'button': {'value': 'unsure', 'icon': 'jba:unsure', 'hot_key': 'u'}},
    ]}},
    {'checkbox': {'label': 'flag for expert', 'column': 'flagged'}},
]

---
## 1. Default: form below the spectrogram

Top to bottom: playlist (info card, filter, clip table), spectrogram, form.

In [3]:
jba(form_config={
    'title': {'value': 'QUICK REVIEW', 'progress_tracker': True},
    'form': QUICK_REVIEW,
});

<IPython.core.display.Javascript object>

---
## 2. Side form

`side_form: true` at the top of the form config puts the form beside the spectrogram. It suits
small forms — a few buttons, a select, a checkbox. The spectrogram takes `side_form_split` of the
width (default 2/3); drag the divider to change it for the session.

- Collapse the **form** (▾ on its title): it becomes a narrow strip with ▸ and the title running vertically; the spectrogram takes the rest.
- Collapse the **spectrogram** (▾ on the play row): it shrinks to the play row; the form takes the rest.

### 2.a Default split (2/3 : 1/3)

In [6]:
jba(form_config={
    'side_form': True,
    'title': {'value': 'QUICK REVIEW', 'progress_tracker': True},
    'form': QUICK_REVIEW,
});

<IPython.core.display.Javascript object>

### 2.b Half and half, with a species select

`side_form_split: 0.5`, and a select with pinned hot keys.

In [9]:
jba(form_config={
    'side_form': True,
    'side_form_split': 0.5,
    'title': {'value': 'SPECIES', 'progress_tracker': True},
    'form': [
        {'select': {'label': 'species', 'column': 'species', 'required': True,
                    'hot_keys': {1: 'Tree Swallow', 2: 'Spotted Towhee', 3: 'Song Sparrow'},
                    'pin_hot_keys': True,
                    'items': {'path': 'data/species_counts.N-dse.csv', 'value': 'common_name',
                              'filter_box': True}}},
        {'textbox': {'label': 'notes', 'multiline': True}},
    ],
});

<IPython.core.display.Javascript object>

### 2.c Wide spectrogram, narrow button column

`side_form_split: 0.8` and a `column` of buttons: one click picks a call type and submits. `submission_buttons: false` drops the separate Submit row.

In [8]:
jba(form_config={
    'side_form': True,
    'side_form_split': 0.8,
    'submission_buttons': False,    # the buttons submit: no Submit row
    'title': 'CALL TYPE',
    'form': [
        {'column': {'column': 'call_type', 'gap': 4, 'items': [
            {'button': {'value': 'song', 'icon': 'song', 'width': 70, 'hot_key': '1', 'submit': True}},
            {'button': {'value': 'call', 'icon': 'call', 'width': 70, 'hot_key': '2', 'submit': True}},
            {'button': {'value': 'other', 'icon': 'other', 'width': 70, 'hot_key': '3', 'submit': True}},
            {'button': {'value': 'noise', 'icon': 'jba:na', 'width': 70, 'hot_key': '0', 'submit': True}},
        ]}},
    ],
});

<IPython.core.display.Javascript object>

### 2.d Thin button column: is it correct?

A very thin form on the right: `side_form_split: 0.93` and a `column` of three one-click buttons
on one output column, `is_correct`: ✓ writes `true`, ✗ `false`, ? `None` (empty). Each submits
and moves on (`submission_buttons: false`); hot keys `y` / `n` / `u` while the column has focus
(click it, or Tab into the form). The split can go up to 0.95; drag the divider to try others.

In [ ]:
jba(form_config={
    'side_form': True,
    'side_form_split': 0.93,
    'submission_buttons': False,
    'form': [
        {'column': {'column': 'is_correct', 'gap': 6, 'items': [
            {'button': {'value': True, 'icon': 'jba:check', 'label': 'correct', 'hot_key': 'y', 'submit': True}},
            {'button': {'value': False, 'icon': 'jba:x', 'label': 'not correct', 'hot_key': 'n', 'submit': True}},
            {'button': {'value': None, 'icon': 'jba:unsure', 'label': 'uncertain', 'hot_key': 'u', 'submit': True}},
        ]}},
    ],
});

---
## 3. Focused review

Everything on one screen: no info card, the playlist collapsed (its ▾ still shows), and the
form beside the spectrogram. Move through clips with `k` / `j`, answer with `y` / `n` / `u`
(Tab into the form first).

In [ ]:
jba(
    hide_info_card=True,
    playlist_open=False,
    playlist_title=True,
    form_config={
        'side_form': True,
        'title': {'value': 'REVIEW [[common_name]]', 'progress_tracker': True},
        'form': QUICK_REVIEW,
    },
);

---
## 4. Starting collapsed

Sections can start collapsed (`playlist_open`, `spectrogram_open`, `form_open`) and show a title
next to their ▾ (`playlist_title`, `spectrogram_title`; `True` gives the standard name).

### 4.a Below: form and playlist collapsed

In [ ]:
jba(
    playlist_open=False,
    form_open=False,
    playlist_title=True,
    spectrogram_title='Spectrogram (kHz)',
    form_config={
        'title': {'value': 'QUICK REVIEW', 'progress_tracker': True},
        'form': QUICK_REVIEW,
    },
);

### 4.b Side form starting as a strip

In [ ]:
jba(
    form_open=False,
    form_config={
        'side_form': True,
        'title': 'QUICK REVIEW',
        'form': QUICK_REVIEW,
    },
);

---
## 5. Form layouts: row / column / grid

`row`, `column` and `grid` arrange elements inside the form (and nest). They work below the
spectrogram and in a side form.

### 5.a Below: a survey form on a grid

In [ ]:
jba(form_config={
    'title': {'value': 'SURVEY', 'progress_tracker': True},
    'form': [
        {'grid': {'columns': 3, 'gap': 12, 'items': [
            {'select': {'label': 'species', 'items': {'path': 'data/species_counts.N-dse.csv',
                                                       'value': 'common_name', 'filter_box': True}}},
            {'number': {'label': 'count', 'min': 0, 'max': 50, 'step': 1, 'value': 1}},
            {'checkbox': {'label': 'juvenile'}},
            {'multiselect': {'label': 'behaviours', 'items': ['singing', 'calling', 'foraging', 'flying']}},
            {'textbox': {'label': 'observer', 'width': 140}},
        ]}},
        {'row': [
            {'text': 'confidence:'},
            {'row': {'column': 'confidence', 'gap': 4, 'items': [
                {'button': {'value': 'low', 'icon': 'L'}},
                {'button': {'value': 'medium', 'icon': 'M'}},
                {'button': {'value': 'high', 'icon': 'H'}},
            ]}},
        ]},
    ],
});

### 5.b Side form: a stacked column of rows

A `column` of `row`s keeps each question on its own line in a narrow side form; the call types
allow several (`unique: false`).

In [ ]:
jba(form_config={
    'side_form': True,
    'side_form_split': 0.80,
    'title': {'value': 'DETAILS', 'progress_tracker': True},
    'form': [
        {'column': {'gap': 10, 'items': [
            {'row': {'column': 'is_valid', 'items': [
                {'text': 'valid?'},
                {'button': {'value': 'yes', 'icon': 'jba:check'}},
                {'button': {'value': 'no', 'icon': 'jba:x'}},
            ]}},
            {'row': [
                {'text': 'calls:'},
                {'grid': {'columns': 2, 'column': 'call_types', 'unique': False, 'delimiter': ';',
                          'items': [
                              {'button': {'value': 'song', 'icon': 'song', 'width': 50}},
                              {'button': {'value': 'call', 'icon': 'call', 'width': 50}},
                              {'button': {'value': 'alarm', 'icon': 'alarm', 'width': 50}},
                              {'button': {'value': 'flight', 'icon': 'flight', 'width': 50}},
                          ]}},
            ]},
            {'checkbox': {'label': 'overlapping', 'column': 'overlapping'}},
        ]}},
    ],
});

---
## 6. Narrow widgets

When the widget is narrower than about 640px, a side form goes back below the spectrogram (no
divider). Widen the cell output (or the widget's resize handle) to bring it back beside it.

In [ ]:
jba(
    width=560,
    form_config={
        'side_form': True,
        'title': 'QUICK REVIEW',
        'form': QUICK_REVIEW,
    },
);